# 04 · Analysis and Statistics

**Goal: turn a pile of readings into numbers you can defend.**

This notebook runs entirely offline. It generates a realistic Beehive dataset,
then computes summary statistics, converts units, reads oversized files without
running out of memory, and audits sensor quality. When you have a real CSV from
notebook `02` or `03`, point the loader at it instead of the synthetic frame and
everything else is unchanged.

A theme runs through this whole notebook: keep measured facts and your reading of
those facts in separate sentences. A statistic is a fact. "The sensor is broken"
is an interpretation. Good analysis never blurs the two.

In [ ]:
# Offline-safe synthetic data generator.
#
# This produces records in the exact schema that sage_data_client.query returns,
# so every SageTools helper (analyzeFile, plotTemperature, auditFile) works on it
# unchanged. Use it when the classroom has no route to the Sage Data API. When
# you do have connectivity, replace the synthetic frame with a real query from
# notebook 02 or 03 and the rest of the notebook is identical.
#
# Nothing here is hidden or disguised: the function is named for what it is, the
# timestamps are fixed to mid 2024, and the values are generated, not measured.

import numpy as np
import pandas as pd


def makeSyntheticBeehive(vsn="W07C", days=14, stepMinutes=5, seed=7,
                         sensors=("bme680", "bme280")):
    """
    Build a Beehive-schema temperature DataFrame with a realistic diurnal
    signal, seasonal drift, per-sensor offsets, and a couple of outage gaps.

    Args:
        vsn: node call sign to stamp on every row.
        days: length of the record in days.
        stepMinutes: spacing between readings, in minutes.
        seed: random seed, so runs are reproducible.
        sensors: one or more sensor names to emit, each with a small bias.

    Returns:
        A DataFrame with columns timestamp, name, value, meta.vsn, meta.sensor,
        meta.host. Values are in Celsius, matching the real feed.
    """
    rng = np.random.default_rng(seed)
    periods = int(days * 24 * 60 / stepMinutes)
    times = pd.date_range("2024-06-01", periods=periods, freq=f"{stepMinutes}min", tz="UTC")

    hourOfDay = times.hour + times.minute / 60.0
    diurnal = 8.0 * np.sin((hourOfDay - 9.0) / 24.0 * 2 * np.pi)  # afternoon peak
    dayIndex = (times - times[0]).days
    seasonalDrift = 0.15 * dayIndex                               # slow warming
    baseline = 21.0

    frames = []
    for offsetIdx, sensor in enumerate(sensors):
        sensorBias = 0.6 * offsetIdx                             # each sensor reads slightly high
        noise = rng.normal(0, 0.4, size=periods)
        celsius = baseline + diurnal + seasonalDrift + sensorBias + noise
        frame = pd.DataFrame({
            "timestamp": times,
            "name": "env.temperature",
            "value": celsius,
            "meta.vsn": vsn,
            "meta.sensor": sensor,
            "meta.host": f"000048b02d{15 + offsetIdx}a.ws-nxcore",
        })
        frames.append(frame)

    combined = pd.concat(frames, ignore_index=True).sort_values("timestamp").reset_index(drop=True)

    # Punch two realistic outage gaps into the record.
    for gapStart, gapHours in [(3, 6), (9, 14)]:
        lo = times[0] + pd.Timedelta(days=gapStart)
        hi = lo + pd.Timedelta(hours=gapHours)
        combined = combined[~combined["timestamp"].between(lo, hi)].reset_index(drop=True)

    return combined

In [ ]:
# Import the SageTools helper package (github.com/mpapka/SageTools).
#
# SageTools is a convenience layer built on top of the official sage_data_client.
# It is optional. Every lab in this course also shows the underlying
# sage_data_client call directly, so the helper is never a hard dependency.
#
# The try/except means the notebook runs whether or not the package is installed:
# if it is missing, haveSageTools becomes False and the notebook falls back to
# the direct calls it already teaches. To install it, see notebook 00.
try:
    import sage
    haveSageTools = True
except ImportError:
    haveSageTools = False

print("SageTools helper package available:", haveSageTools)

## 1. Build a dataset to analyze

The generator produces two temperature sensors on one node over two weeks, with a
daily cycle, a slow warming trend, a per sensor bias, and two outage gaps. We save
it with the node's call sign in the filename, because the SageTools analysis tools
recover the call sign from the filename (they split on the underscore and strip
the extension, so `chicago_temp_W06C.csv` yields `W06C`).

In [ ]:
sampleDf = makeSyntheticBeehive(vsn="W06C", days=14)
csvPath = "chicago_temp_W06C.csv"
sampleDf.to_csv(csvPath, index=False)

print("rows:", len(sampleDf))
print("sensors:", sorted(sampleDf["meta.sensor"].unique()))
print("time span:", sampleDf["timestamp"].min(), "->", sampleDf["timestamp"].max())
sampleDf.head()

### Swapping in real data

To analyze real measurements instead, replace the two cells above with a load of
a CSV you saved in notebook `02` or `03`:

```python
sampleDf = pd.read_csv("chicago_temp_W06C.csv", low_memory=False)
sampleDf["timestamp"] = pd.to_datetime(sampleDf["timestamp"])
```

Because the synthetic frame uses the exact real schema, nothing downstream needs
to change.

## 2. Units: Celsius to Fahrenheit

The Beehive reports temperature in Celsius. Whether you present Celsius or
Fahrenheit is an audience choice, but be explicit about which you used, every
time. The conversion is `F = C * 9/5 + 32`.

In [ ]:
sampleDf["valueF"] = sampleDf["value"] * 9 / 5 + 32
sampleDf[["timestamp", "value", "valueF", "meta.sensor"]].head()

## 3. Summary statistics

The first questions about any temperature record: what were the extremes, and
what was typical? Always report the count alongside the statistics. A maximum
computed from three readings is a much weaker claim than one from thirty thousand,
and the count is what lets your reader tell the difference.

In [ ]:
def summarize(frame, valueCol="value"):
    """Return basic temperature statistics in both Celsius and Fahrenheit."""
    celsius = pd.to_numeric(frame[valueCol], errors="coerce").dropna()
    stats = {
        "count": int(celsius.count()),
        "minC": float(celsius.min()),
        "maxC": float(celsius.max()),
        "meanC": float(celsius.mean()),
        "stdC": float(celsius.std()),
    }
    stats["minF"] = stats["minC"] * 9 / 5 + 32
    stats["maxF"] = stats["maxC"] * 9 / 5 + 32
    stats["meanF"] = stats["meanC"] * 9 / 5 + 32
    return stats


summary = summarize(sampleDf)
for key, val in summary.items():
    if isinstance(val, float):
        print(f"{key:>7}: {val:,.2f}")
    else:
        print(f"{key:>7}: {val:,}")

### Fact and interpretation, kept apart

State measured facts and your reading of them in separate sentences. This habit
is the difference between analysis and storytelling.

- **Fact.** Over this window the mean was the value printed above, with the
  minimum and maximum as shown, across the reported record count.
- **Interpretation.** Any claim about "a heat wave" or "a failing sensor" is a
  hypothesis, not a measurement. Label it as such, and test it against more data
  before you rely on it.

## 4. Reading oversized files in chunks

A season of data sampled every few seconds can be a multi gigabyte CSV that will
not fit in memory. `pandas.read_csv` accepts a `chunksize` argument, which yields
the file a piece at a time. You accumulate running statistics across the pieces
and never hold the whole file at once. This is exactly how SageTools' `analyzeFile`
handles large records.

Notice the running statistics pattern: min and max are updated per chunk, and the
mean is computed at the end from a running sum and a running count, never from all
the values at once.

In [ ]:
def summarizeLargeFile(path, chunkSize=500_000):
    """Compute min, max, and mean over a CSV too large to load at once."""
    minVal, maxVal = float("inf"), float("-inf")
    runningSum, runningCount = 0.0, 0
    sensorsSeen = set()

    for chunk in pd.read_csv(path, usecols=["value", "meta.sensor"],
                             chunksize=chunkSize, low_memory=False,
                             on_bad_lines="skip"):
        vals = pd.to_numeric(chunk["value"], errors="coerce").dropna()
        if vals.empty:
            continue
        sensorsSeen.update(chunk["meta.sensor"].dropna().unique())
        minVal = min(minVal, vals.min())
        maxVal = max(maxVal, vals.max())
        runningSum += vals.sum()
        runningCount += len(vals)

    return {
        "count": runningCount,
        "minC": minVal,
        "maxC": maxVal,
        "meanC": runningSum / runningCount if runningCount else None,
        "sensors": sorted(sensorsSeen),
    }


print(summarizeLargeFile(csvPath))

In [ ]:
# The SageTools equivalent reports in Fahrenheit and prints a formatted table.
if haveSageTools:
    from sage.beehive.analysis.statistics import analyzeFile
    print(analyzeFile(csvPath))

## 5. Sensor auditing

A node often carries more than one temperature sensor. When they disagree, one may
be failing, or one may sit near a heat source on the circuit board and read
persistently high. Auditing means computing per sensor statistics and looking for
a sensor whose behavior stands out. Below we compare the two sensors in our
dataset.

In [ ]:
audit = (
    sampleDf
    .assign(valueF=sampleDf["value"] * 9 / 5 + 32)
    .groupby("meta.sensor")["valueF"]
    .agg(["count", "min", "max", "mean", "std"])
    .round(2)
)
print(audit)

meanSpread = audit["mean"].max() - audit["mean"].min()
print(f"\nmean spread between sensors: {meanSpread:.2f} F")
print()
print("Fact: the two sensors differ in mean by the spread above.")
print("Interpretation to test: a persistent offset can indicate board heating, a")
print("wiring fault, or simply two sensors of different make. Confirm the cause")
print("with more data before discarding either stream.")

In [ ]:
# The SageTools sensor audit needs a meta.host column, which the synthetic frame
# includes, and it flags suspicious sensors automatically.
if haveSageTools:
    from sage.beehive.analysis.sensorAudit import auditFile
    for record in auditFile(csvPath):
        print(record)

## 6. Cleaning bad readings before you summarize

Every statistic above assumed the values were real. Real sensor feeds break that
assumption. A disconnected probe reports a flat 0, a firmware fault emits a code
like -200, a sun struck sensor spikes high. One -200 mixed into readings near 70
drags the mean down and makes the minimum meaningless, so you remove obvious
errors before you summarize.

Two filters catch different problems.

- A **physical plausibility band** drops values outside the hard limits of the
  metric. This catches gross errors like -200 and 999. Set the band to your
  metric; for ambient air temperature in Fahrenheit, roughly -40 to 140 works.
- A **per-node robust outlier check** drops values far from the node's own median,
  scaled by the median absolute deviation (MAD). The MAD is itself resistant to
  the errors it is finding, and it adapts to a node's normal level, so it catches
  a flat 0 among readings near 70 without hardcoding a range. The default cutoff
  of 3.5 is the standard modified z-score threshold. See Further reading for the
  MAD.

In [ ]:
def dropImplausible(frame, valueCol="valueF", minValid=-40.0, maxValid=140.0):
    """
    Remove readings outside a physically plausible band.

    The band is a hard physical limit for the metric, not a statistical one. Set
    it to match your metric. Returns (cleanedFrame, removedCount).
    """
    numeric = pd.to_numeric(frame[valueCol], errors="coerce")
    keep = numeric.between(minValid, maxValid)
    return frame.loc[keep].copy(), int((~keep).sum())


def dropOutliersMad(frame, valueCol="valueF", threshold=3.5):
    """
    Remove readings far from the node's own median, using the median absolute
    deviation (MAD). The default 3.5 is the standard modified z-score cutoff,
    aggressive enough to catch a flat 0 among readings near 70 while keeping real
    daily extremes. Returns (cleanedFrame, removedCount).
    """
    numeric = pd.to_numeric(frame[valueCol], errors="coerce")
    median = numeric.median()
    mad = (numeric - median).abs().median()
    if mad == 0:
        return frame.copy(), 0
    scaledMad = 1.4826 * mad
    keep = (numeric - median).abs() <= threshold * scaledMad
    return frame.loc[keep].copy(), int((~keep).sum())


def injectErrors(frame, valueCol="valueF", n=25, seed=0):
    """Insert obvious sensor-error values into a copy of a frame, for teaching."""
    if len(frame) == 0:
        return frame.copy()
    rng = np.random.default_rng(seed)
    dirty = frame.copy().reset_index(drop=True)
    errorValues = np.array([0.0, -50.0, -200.0, 999.0])
    idx = rng.choice(dirty.index, size=min(n, len(dirty)), replace=False)
    dirty.loc[idx, valueCol] = rng.choice(errorValues, size=len(idx))
    return dirty


print("filters defined: dropImplausible, dropOutliersMad, injectErrors")

### See it: the same node before and after filtering

Counts are informative but a picture is clearer. We take one sensor's clean
readings, inject obvious errors, then filter them out. The left panel shows every
raw reading with the removed values marked as red crosses; the error spikes
stretch the axis so far that the real daily signal is squeezed into a thin band.
The right panel shows only the surviving readings, where the pattern is legible
again.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

# One sensor, with a Fahrenheit column, then inject errors to remove.
oneSensor = sampleDf[sampleDf["meta.sensor"] == "bme680"].copy()
oneSensor["valueF"] = oneSensor["value"] * 9 / 5 + 32
oneSensor["timestamp"] = pd.to_datetime(oneSensor["timestamp"])
oneSensor = oneSensor.sort_values("timestamp").reset_index(drop=True)

dirtyDf = injectErrors(oneSensor, valueCol="valueF", n=25, seed=1)
print(f"raw: {len(dirtyDf)} readings, "
      f"range {dirtyDf['valueF'].min():.1f} to {dirtyDf['valueF'].max():.1f} F")

step1Df, removedBand = dropImplausible(dirtyDf, "valueF", minValid=-40.0, maxValid=140.0)
cleanDf, removedMad = dropOutliersMad(step1Df, "valueF", threshold=3.5)
removedDf = dirtyDf.loc[~dirtyDf.index.isin(cleanDf.index)]
print(f"physical band removed {removedBand}, MAD removed {removedMad}, "
      f"{len(removedDf)} removed in total")
print(f"clean: {len(cleanDf)} readings, "
      f"range {cleanDf['valueF'].min():.1f} to {cleanDf['valueF'].max():.1f} F")

fig, (axBefore, axAfter) = plt.subplots(1, 2, figsize=(14, 5))

axBefore.plot(dirtyDf["timestamp"], dirtyDf["valueF"], ".", markersize=3,
              color="tab:blue", label="kept")
if not removedDf.empty:
    axBefore.plot(removedDf["timestamp"], removedDf["valueF"], "x", markersize=8,
                  color="red", label="removed as error")
axBefore.set_title(f"before filtering: {len(dirtyDf)} readings")
axBefore.set_xlabel("Time")
axBefore.set_ylabel("Temperature (F)")
axBefore.legend(loc="upper right")
axBefore.grid(True, linestyle="--", alpha=0.5)
axBefore.xaxis.set_major_formatter(mdates.DateFormatter("%b %d"))

axAfter.plot(cleanDf["timestamp"], cleanDf["valueF"], ".", markersize=3, color="tab:green")
axAfter.set_title(f"after filtering: {len(cleanDf)} readings")
axAfter.set_xlabel("Time")
axAfter.set_ylabel("Temperature (F)")
axAfter.grid(True, linestyle="--", alpha=0.5)
axAfter.xaxis.set_major_formatter(mdates.DateFormatter("%b %d"))

for ax in (axBefore, axAfter):
    for tickLabel in ax.get_xticklabels():
        tickLabel.set_rotation(45)

plt.tight_layout()
plt.show()

## 7. Exercises

1. Regenerate the dataset with `days=30` and recompute the summary. Which
   statistics move the most as the window lengthens, and which barely change?
2. Add a median to `summarize`. When does the median tell a more honest story than
   the mean?
   *Hint: think about a single stuck sensor reporting a wild value.*
3. The audit shows a mean spread between the two sensors. Write exactly two
   sentences: one stating the measured spread, one proposing a testable
   explanation. Keep them separate.
4. Resample the data to hourly means, then recompute the statistics. How does the
   minimum to maximum range change, and why does smoothing shrink it?
   *Hint: set `timestamp` as the index, then use `resample("1h").mean()`.*
5. Re-run the filters with a MAD threshold of 6.0 instead of 3.5. Which
   injected errors now survive, and why does a flat 0 slip through a lenient
   threshold when readings sit near 70?

## Further reading

- Median absolute deviation, the basis of the robust outlier filter: https://en.wikipedia.org/wiki/Median_absolute_deviation
- Robust statistics, why the median and MAD resist outliers: https://en.wikipedia.org/wiki/Robust_statistics
- Detection and treatment of outliers, NIST handbook: https://www.itl.nist.gov/div898/handbook/
- pandas `groupby` for per-sensor and per-node statistics: https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.groupby.html